In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
!mkdir images_train
!mkdir images_train/0
!mkdir images_train/1
!mkdir images_train/2
!mkdir images_train/3
!mkdir images_train/4
!mkdir images_train/5
!mkdir images_train/6
!mkdir images_train/7
!mkdir images_train/8
!mkdir images_train/9
!mkdir images_test

In [ ]:
csv_train = pd.read_csv('./drive/MyDrive/dacon/train.csv')
csv_train.head()
for idx in range(len(csv_train)) :
    img = csv_train.loc[idx, '0':].values.reshape(28, 28).astype(int)
    digit = csv_train.loc[idx, 'digit']
    cv2.imwrite(f'./images_train/{digit}/{csv_train["id"][idx]}.png', img)



In [ ]:
import albumentations as A
augmentor_01 = A.Compose([
    # A.HorizontalFlip(p=0.5),
    # A.VerticalFlip(p=0.5),
    # A.ShiftScaleRotate(scale_limit=(0.7, 0.9), p=0.5, rotate_limit=30),
    A.RandomBrightnessContrast(brightness_limit=(-0.2, 0.2), contrast_limit=(-0.2, 0.2), p=0.5),
    A.Blur(p=0.2)
])

In [ ]:
from tensorflow.keras.models import Sequential , Model
from tensorflow.keras.layers import Input, Dense , Conv2D , Dropout , Flatten , Activation, MaxPooling2D , GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam , RMSprop 
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau , EarlyStopping , ModelCheckpoint , LearningRateScheduler
from tensorflow.keras.metrics import AUC

from tensorflow.keras.applications import Xception, ResNet50V2, EfficientNetB0, EfficientNetB1, EfficientNetB2, EfficientNetB3
from tensorflow.keras.applications import EfficientNetB4, EfficientNetB5, EfficientNetB6, EfficientNetB7
import tensorflow as tf


def create_model(model_type='efficientnetb0', in_shape=(224, 224), n_classes=10):
    input_tensor = Input(shape=in_shape)

    if model_type == 'resnet50v2':
        base_model = tf.keras.applications.ResNet50V2(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'xception':
        base_model = tf.keras.applications.Xception(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb0':
        base_model = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb1':
        base_model = tf.keras.applications.EfficientNetB1(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb2':
        base_model = tf.keras.applications.EfficientNetB2(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb3':
        base_model = tf.keras.applications.EfficientNetB3(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb4':
        base_model = tf.keras.applications.EfficientNetB4(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb5':
        base_model = tf.keras.applications.EfficientNetB5(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb6':
        base_model = tf.keras.applications.EfficientNetB6(include_top=False, weights='imagenet', input_tensor=input_tensor)
    elif model_type == 'efficientnetb7':
        base_model = tf.keras.applications.EfficientNetB7(include_top=False, weights='imagenet', input_tensor=input_tensor)
        
    x = base_model.output  
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)    
    preds = Dense(units=n_classes, activation='softmax')(x)
    model = Model(inputs=input_tensor, outputs=preds)
    

    return model

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import Sequence
import sklearn 
import cv2

class NumDataSet(Sequence):
    def __init__(self, image_filenames, labels, image_size=(224, 224), batch_size=64, 
                 augmentor=None, shuffle=False, pre_func=None):
      
        # 객체 생성 인자로 들어온 값을 객체 내부 변수로 할당. 
        self.image_filenames = image_filenames
        self.labels = labels
        self.image_size = image_size
        self.batch_size = batch_size
        self.augmentor = augmentor
        self.pre_func = pre_func
        # train data의 경우 
        self.shuffle = shuffle
        if self.shuffle:
            # 객체 생성시에 한번 데이터를 섞음. 
            self.on_epoch_end()
            # pass
    
    # Sequence를 상속받은 Dataset은 batch_size 단위로 입력된 데이터를 처리함. 
    # __len__()은 전체 데이터 건수가 주어졌을 때 batch_size단위로 몇번 데이터를 반환하는지 나타남
    def __len__(self):
        # batch_size단위로 데이터를 몇번 가져와야하는지 계산하기 위해 전체 데이터 건수를 batch_size로 나누되, 정수로 정확히 나눠지지 않을 경우 1회를 더한다. 
        return int(np.ceil(len(self.image_filenames) / self.batch_size))
    
    # batch_size 단위로 image_array, label_array 데이터를 가져와서 변환한 뒤 다시 반환함
    # 인자로 몇번째 batch 인지를 나타내는 index를 입력하면 해당 순서에 해당하는 batch_size 만큼의 데이타를 가공하여 반환
    # batch_size 갯수만큼 변환된 image_array와 label_array 반환. 
    def __getitem__(self, index):
        # index는 몇번째 batch인지를 나타냄. 
        # batch_size만큼 순차적으로 데이터를 가져오려면 array에서 index*self.batch_size:(index+1)*self.batch_size 만큼의 연속 데이터를 가져오면 됨
        image_name_batch = self.image_filenames[index*self.batch_size:(index+1)*self.batch_size]
        if self.labels is not None:
            label_batch = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        # label_batch가 None이 될 수 있음. 
        else: 
            label_batch = None
        # 만일 객체 생성 인자로 albumentation으로 만든 augmentor가 주어진다면 아래와 같이 augmentor를 이용하여 image 변환
        # albumentations은 개별 image만 변환할 수 있으므로 batch_size만큼 할당된 image_name_batch를 한 건씩 iteration하면서 변환 수행. 
        # image_batch 배열은 float32 로 설정. 
        image_batch = np.zeros((image_name_batch.shape[0], self.image_size[0], self.image_size[1], 3), dtype='float32')
        
        # batch_size에 담긴 건수만큼 iteration 하면서 opencv image load -> image augmentation 변환(augmentor가 not None일 경우)-> image_batch에 담음. 
        for image_index in range(image_name_batch.shape[0]):
            image = cv2.imread(image_name_batch[image_index])
            # print(image)
            # image = cv2.cvtColor(cv2.imread(image_name_batch[image_index]), cv2.COLOR_BGR2RGB)
            if self.augmentor is not None:
                image = self.augmentor(image=image)['image']
            #원본 이미지와 다르게 resize 적용. opencv의 resize은 (가로, 세로)의 개념임. 배열은 (높이, 너비)의 개념이므로 이에 주의하여 opencv resize 인자 입력 필요.  
            image = cv2.resize(image, (self.image_size[1], self.image_size[0]))
            # 만일 preprocessing_input이 pre_func인자로 들어오면 이를 이용하여 scaling 적용. 
            if self.pre_func is not None:
                image = self.pre_func(image)
            # image.reshape()
            
            image_batch[image_index] = image
        
        return image_batch, label_batch
    
    # epoch가 한번 수행이 완료 될 때마다 모델의 fit()에서 호출됨. 
    def on_epoch_end(self):
        if(self.shuffle):
            #print('epoch end')
            # 전체 image 파일의 위치와 label를 쌍을 맞춰서 섞어준다. scikt learn의 utils.shuffle에서 해당 기능 제공
            self.image_filenames, self.labels = sklearn.utils.shuffle(self.image_filenames, self.labels)
        else:
            pass

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import Sequence
import sklearn 
import cv2

import albumentations as A
IMAGE_DIR='./images_train'
def MakeDataFrame(image_dir=IMAGE_DIR):
  paths = []
  label_data = []
  for dirname, _, filenames in os.walk(image_dir):
    for filename in filenames:
      if '.png' in filename:
        file_path = dirname+'/'+filename
        paths.append(file_path)
        start_point = file_path.find('/',5)
        end_point = file_path.rfind('/')
        number_tmp = file_path[start_point+1:end_point]
        # label_num.append(number_tmp[start_point+1:])
        label_data.append(number_tmp)
        # print
  data_df = pd.DataFrame({'path':paths, 'label':label_data})
  return data_df

In [ ]:
from sklearn.model_selection import train_test_split

def get_train_valid(train_df, valid_size=0.2, random_state=2021):
    train_path = train_df['path'].values
    # 별도의 원핫인코딩을 하지 않고  'healthy', 'multiple_diseases', 'rust', 'scab' 컬럼들을 모두 Numpy array로 변환하는 수준으로 label을 원핫 인코딩 적용. 
    train_label = pd.get_dummies(train_df['label']).values    
    tr_path, val_path, tr_label, val_label = train_test_split(train_path, train_label, test_size=valid_size, random_state=random_state)
    # print('tr_path shape:', tr_path.shape, 'tr_label shape:', tr_label.shape, 'val_path shape:', val_path.shape, 'val_label shape:', val_label.shape)
    return tr_path, val_path, tr_label, val_label

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess_input
from tensorflow.keras.applications.xception import preprocess_input as xcp_preprocess_input
import tensorflow as tf

# learning rate scheduler에 적용할 함수 선언. 
def lrfn_01(epoch):
    LR_START = 1e-5
    LR_MAX = 1e-4
    LR_RAMPUP_EPOCHS = 2
    LR_SUSTAIN_EPOCHS = 1
    LR_STEP_DECAY = 0.75
    
    def calc_fn(epoch):
        if epoch < LR_RAMPUP_EPOCHS:
            lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch + LR_START
        elif epoch < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
            lr = LR_MAX
        else:
            lr = LR_MAX * LR_STEP_DECAY**((epoch - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS)//2)
        return lr
    
    return calc_fn(epoch)

def lrfn_02(epoch):
    LR_START = 1e-6
    LR_MAX = 2e-5
    LR_RAMPUP_EPOCHS = 2
    LR_SUSTAIN_EPOCHS = 1
    LR_STEP_DECAY = 0.75
    
    def calc_fn(epoch):
        if epoch < LR_RAMPUP_EPOCHS:
            lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch + LR_START
        elif epoch < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
            lr = LR_MAX
        else:
            lr = LR_MAX * LR_STEP_DECAY**((epoch - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS)//2)
        return lr
    
    return calc_fn(epoch)


In [ ]:
from tensorflow.keras.applications.xception import preprocess_input as xcp_preprocess_input
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess_input

# IMAGE_DIR='./images_train'
# tmp_df = MakeDataFrame(IMAGE_DIR)
# tmp_df.head()

# tmp_df.index.values

# image size는 224x224로 Dataset 생성. 
IMAGE_SIZE_1 = (224, 224,3)
IMAGE_SIZE_2 = (300,300,3)
IMAGE_SIZE_3=(400,400,3)

BATCH_SIZE_1 = 64
BATCH_SIZE_2 = 5
BATCH_SIZE_3 = 2

data_df = MakeDataFrame(image_dir=IMAGE_DIR)
tr_path, val_path, tr_label, val_label = get_train_valid(data_df, valid_size=0.2, random_state=2021)

tr_ds_1 = NumDataSet(tr_path, tr_label, image_size=IMAGE_SIZE_1, batch_size=BATCH_SIZE_1, 
                          augmentor=augmentor_01, shuffle=True, pre_func=xcp_preprocess_input)
val_ds_1 = NumDataSet(val_path, val_label, image_size=IMAGE_SIZE_1, batch_size=BATCH_SIZE_1, 
                      augmentor=None, shuffle=False, pre_func=xcp_preprocess_input)

tr_ds_2 = NumDataSet(tr_path, tr_label, image_size=IMAGE_SIZE_2, batch_size=BATCH_SIZE_2, 
                          augmentor=augmentor_01, shuffle=True, pre_func=xcp_preprocess_input)
val_ds_2 = NumDataSet(val_path, val_label, image_size=IMAGE_SIZE_2, batch_size=BATCH_SIZE_2, 
                      augmentor=None, shuffle=False, pre_func=xcp_preprocess_input)

tr_ds_3 = NumDataSet(tr_path, tr_label, image_size=IMAGE_SIZE_3, batch_size=BATCH_SIZE_3, 
                          augmentor=augmentor_01, shuffle=True, pre_func=xcp_preprocess_input)
val_ds_3 = NumDataSet(val_path, val_label, image_size=IMAGE_SIZE_3, batch_size=BATCH_SIZE_3, 
                      augmentor=None, shuffle=False, pre_func=xcp_preprocess_input)

# tr_image_batch, tr_label_batch = next(iter(tr_ds))
# val_image_batch, val_label_batch = next(iter(val_ds))
# print(tr_image_batch.shape, val_image_batch.shape, tr_label_batch.shape, val_label_batch.shape)
# print(tr_image_batch[0], val_image_batch[0])

In [ ]:

model1 = create_model(model_type='xception', in_shape=IMAGE_SIZE_1)
model1.compile(optimizer=Adam(lr=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

model2 = create_model(model_type='efficientnetb3', in_shape=IMAGE_SIZE_2)
model2.compile(optimizer=Adam(lr=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

model3 = create_model(model_type='efficientnetb7', in_shape=IMAGE_SIZE_3)
model3.compile(optimizer=Adam(lr=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

lr01_cb = tf.keras.callbacks.LearningRateScheduler(lrfn_01, verbose=1)
lr02_cb = tf.keras.callbacks.LearningRateScheduler(lrfn_02, verbose=1)
rlr_cb = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, mode='min', verbose=1)

ely_cb = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=1)

check_cb_1 =  tf.keras.callbacks.ModelCheckpoint(
        filepath = f'./drive/My Drive/dacon/model_1.h5',
        monitor='val_accuracy',
        save_best_only=True
    )

check_cb_2 =  tf.keras.callbacks.ModelCheckpoint(
        filepath = f'./drive/My Drive/dacon/model_2.h5',
        monitor='val_accuracy',
        save_best_only=True
    )

check_cb_3 =  tf.keras.callbacks.ModelCheckpoint(
        filepath = f'./drive/My Drive/dacon/model_3.h5',
        monitor='val_accuracy',
        save_best_only=True
    )



258088960/258076736 [==============================] - 6s 0us/step


In [ ]:

history3 = model3.fit(tr_ds_3, epochs=100, steps_per_epoch=int(np.ceil(tr_path.shape[0]/BATCH_SIZE_3)), 
               validation_data=val_ds_3, validation_steps=int(np.ceil(val_path.shape[0]/BATCH_SIZE_3)),
               callbacks=([lr01_cb, ely_cb,check_cb_3]), verbose=1)


history2 = model2.fit(tr_ds_2, epochs=100, steps_per_epoch=int(np.ceil(tr_path.shape[0]/BATCH_SIZE_2)), 
               validation_data=val_ds_2, validation_steps=int(np.ceil(val_path.shape[0]/BATCH_SIZE_2)),
               callbacks=([lr01_cb, ely_cb,check_cb_2]), verbose=1)




history1 = model1.fit(tr_ds_1, epochs=100, steps_per_epoch=int(np.ceil(tr_path.shape[0]/BATCH_SIZE_1)), 
               validation_data=val_ds_1, validation_steps=int(np.ceil(val_path.shape[0]/BATCH_SIZE_1)),
               callbacks=([lr01_cb, ely_cb,check_cb_1]), verbose=1)






Epoch 00001: LearningRateScheduler setting learning rate to 1e-05.
Epoch 1/100
819/819 [==============================] - 966s 1s/step - loss: 2.3160 - accuracy: 0.0940 - val_loss: 2.2919 - val_accuracy: 0.1195 - lr: 1.0000e-05

Epoch 00002: LearningRateScheduler setting learning rate to 5.5e-05.
Epoch 2/100
819/819 [==============================] - 910s 1s/step - loss: 2.2263 - accuracy: 0.1722 - val_loss: 2.3185 - val_accuracy: 0.0927 - lr: 5.5000e-05

Epoch 00003: LearningRateScheduler setting learning rate to 0.0001.
Epoch 3/100
819/819 [==============================] - 923s 1s/step - loss: 1.8053 - accuracy: 0.3694 - val_loss: 2.3214 - val_accuracy: 0.1512 - lr: 1.0000e-04

Epoch 00004: LearningRateScheduler setting learning rate to 0.0001.
Epoch 4/100
819/819 [==============================] - 925s 1s/step - loss: 1.1633 - accuracy: 0.6172 - val_loss: 2.1067 - val_accuracy: 0.2488 - lr: 1.0000e-04

Epoch 00005: LearningRateScheduler setting learning rate to 0.0001.
Epoch 5/100

In [ ]:
model_1 = tf.keras.models.load_model('./drive/My Drive/dacon/model_1.h5', compile=False)
# model_2 = tf.keras.models.load_model('./drive/My Drive/dacon/model_2.h5', compile=False)
model_3 = tf.keras.models.load_model('./drive/My Drive/dacon/model_3.h5', compile=False)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(model_1.history.history["accuracy"], label='m1_acc')
plt.plot(model_1.history.history["val_accuracy"], label='m1_vacc')

plt.plot(model_2.history.history["accuracy"], label='m2_acc')
plt.plot(model_2.history.history["val_accuracy"], label='m2_vacc')

plt.plot(model_3.history.history["accuracy"], label='m3_acc')
plt.plot(model_3.history.history["val_accuracy"], label='m3_acc')

plt.legend()
plt.show()

AttributeError: ignored